In [ ]:
%pip install openpyxl

## 1. Đọc tập Train và Validation đã chia sẵn


In [ ]:
from datetime import datetime
from pathlib import Path
import inspect
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "Attack_type"

current = Path.cwd().resolve()
SEARCH_ROOTS = [current, *current.parents]
ROOT = next(
    (
        folder
        for folder in SEARCH_ROOTS
        if (folder / "project").is_dir() and (folder / "result").is_dir()
    ),
    None,
)
if ROOT is None:
    ROOT = next(
        (
            candidate
            for folder in SEARCH_ROOTS
            for candidate in (
                folder / "final_IoT" / "sample_project",
                folder / "sample_project",
            )
            if (candidate / "project").is_dir()
            and (candidate / "result").is_dir()
        ),
        None,
    )
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục sample_project")

DATA_DIR = ROOT / "splitted data"
TRAIN_PATH = DATA_DIR / "train.csv.gz"
VALID_PATH = DATA_DIR / "validation.csv.gz"
for path in (TRAIN_PATH, VALID_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy tệp dữ liệu: {path}")

RESULT_DIR = ROOT / "result" / "baseline" / "E10_MLP_Baseline"
RUN_DIR = RESULT_DIR / datetime.now().strftime("E10_MLP_%Y%m%dT%H%M%S%f")
TABLE_DIR = RUN_DIR / "tables"
IMAGE_DIR = RUN_DIR / "images"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)

if TARGET not in train_df.columns or TARGET not in valid_df.columns:
    raise ValueError(f"Cả Train và Validation đều phải có cột nhãn {TARGET!r}")
if not train_df.columns.equals(valid_df.columns):
    raise ValueError("Tên cột hoặc thứ tự cột Train và Validation không khớp")
if train_df.empty or valid_df.empty:
    raise ValueError("Train và Validation không được để trống")

print("Tập Train:", TRAIN_PATH, train_df.shape)
print("Tập Validation:", VALID_PATH, valid_df.shape)
print("Không đọc hoặc sử dụng tập Test trong baseline này")
display(train_df.head(3))

## 2. Chuẩn bị đặc trưng và nhãn



In [ ]:
X_train = train_df.drop(columns=[TARGET]).copy()
y_train = train_df[TARGET].astype(str).copy()
X_valid = valid_df.drop(columns=[TARGET]).copy()
y_valid = valid_df[TARGET].astype(str).copy()

categorical_features = ["proto", "service"]
missing_categories = [col for col in categorical_features if col not in X_train.columns]
if missing_categories:
    raise ValueError(f"Thiếu đặc trưng phân loại: {missing_categories}")
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
if set(numeric_features + categorical_features) != set(X_train.columns):
    raise ValueError("Có đặc trưng chưa được phân loại là số hoặc phân loại")
if X_train.isna().any().any() or X_valid.isna().any().any():
    raise ValueError("Train hoặc Validation có giá trị thiếu")
if not np.isfinite(X_train[numeric_features].to_numpy()).all():
    raise ValueError("Đặc trưng số của Train có giá trị NaN/Inf")
if not np.isfinite(X_valid[numeric_features].to_numpy()).all():
    raise ValueError("Đặc trưng số của Validation có giá trị NaN/Inf")
if not X_train.columns.equals(X_valid.columns):
    raise ValueError("Các cột đặc trưng Train và Validation không khớp")

LABELS = sorted(y_train.unique().tolist())
unknown_labels = sorted(set(y_valid.unique()) - set(LABELS))
if unknown_labels:
    raise ValueError(f"Validation có nhãn không xuất hiện trong Train: {unknown_labels}")

print("Số mẫu Train:", len(X_train))
print("Số mẫu Validation:", len(X_valid))
print("Số đặc trưng số:", len(numeric_features))
print("Đặc trưng phân loại:", categorical_features)
print("Số lớp:", len(LABELS))
display(y_train.value_counts().rename("Train samples").to_frame())

## 3. Xây dựng pipeline MLP Baseline



In [ ]:
ohe_params = {"handle_unknown": "ignore"}
if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
    ohe_params["sparse_output"] = False
else:
    ohe_params["sparse"] = False

METRICS = ["accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall"]

def make_pipeline():
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_features),
            ("categorical", OneHotEncoder(**ohe_params), categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0,
    )
    return Pipeline([
        ("preprocess", preprocessor),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=(64,),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            learning_rate_init=0.001,
            batch_size=512,
            max_iter=60,
            n_iter_no_change=8,
            early_stopping=False,
            random_state=RANDOM_STATE,
            verbose=False,
        )),
    ])

def evaluate(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=LABELS, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
    }

model = make_pipeline()
print("Cấu hình pipeline MLP Baseline:")
print(model)

## 4. Thực hiện 3-fold Cross-Validation trên tập Train có sẵn



In [ ]:
N_SPLITS = 3
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
cv_class_rows = []
cv_loss_curves = {}

for fold, (fit_idx, score_idx) in enumerate(cv.split(X_train, y_train), start=1):
    X_fit, y_fit = X_train.iloc[fit_idx], y_train.iloc[fit_idx]
    X_score, y_score = X_train.iloc[score_idx], y_train.iloc[score_idx]
    cv_model = make_pipeline()
    started = time.perf_counter()
    cv_model.fit(X_fit, y_fit)
    fit_seconds = time.perf_counter() - started
    predictions = cv_model.predict(X_score)
    fit_predictions = cv_model.predict(X_fit)
    scores = evaluate(y_score, predictions)
    fit_scores = evaluate(y_fit, fit_predictions)
    fitted_mlp = cv_model.named_steps["classifier"]
    cv_loss_curves[fold] = list(fitted_mlp.loss_curve_)
    cv_rows.append({
        "fold": fold,
        "train_samples": len(X_fit),
        "fold_validation_samples": len(X_score),
        "fit_seconds": fit_seconds,
        "epochs": fitted_mlp.n_iter_,
        **scores,
        **{f"train_{metric}": value for metric, value in fit_scores.items()},
    })
    precision, recall, f1, support = precision_recall_fscore_support(
        y_score, predictions, labels=LABELS, zero_division=0
    )
    for label, p, r, score, count in zip(LABELS, precision, recall, f1, support):
        cv_class_rows.append({
            "fold": fold,
            "class": label,
            "precision": float(p),
            "recall": float(r),
            "f1": float(score),
            "support": int(count),
        })
    print(f"Fold {fold}/{N_SPLITS}: Macro-F1={scores['macro_f1']:.6f}; số epoch={fitted_mlp.n_iter_}; thời gian huấn luyện={fit_seconds:.2f} giây")

cv_results = pd.DataFrame(cv_rows)
cv_per_class = pd.DataFrame(cv_class_rows)
cv_summary = pd.DataFrame({
    "metric": METRICS,
    "mean": [cv_results[metric].mean() for metric in METRICS],
    "std": [cv_results[metric].std(ddof=1) for metric in METRICS],
})
print("Cross-Validation chỉ sử dụng Train:", len(X_train), "dòng")
display(cv_results[["fold", *METRICS, "fit_seconds", "epochs"]].round(6))
display(cv_summary.round(6))

## 5. Huấn luyện trên toàn bộ Train và đánh giá trên Validation độc lập


In [ ]:
started = time.perf_counter()
model.fit(X_train, y_train)
training_seconds = time.perf_counter() - started

y_train_pred = model.predict(X_train)
y_valid_pred = model.predict(X_valid)
train_metrics = evaluate(y_train, y_train_pred)
valid_metrics = evaluate(y_valid, y_valid_pred)

metrics_df = pd.DataFrame([
    {"split": "train", **train_metrics},
    {"split": "validation", **valid_metrics},
])
print(f"Thời gian huấn luyện: {training_seconds:.2f} giây")
print("Số epoch:", model.named_steps["classifier"].n_iter_)
print("Loss huấn luyện cuối cùng:", f"{model.named_steps['classifier'].loss_:.6f}")
display(metrics_df.round(6))

## 6. Chỉ số theo từng lớp và biểu đồ


In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_valid, y_valid_pred, labels=LABELS, zero_division=0
)
per_class = pd.DataFrame({
    "class": LABELS,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
}).sort_values("f1").reset_index(drop=True)
display(per_class.round(6))

cm = confusion_matrix(y_valid, y_valid_pred, labels=LABELS)
row_totals = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(
    cm, row_totals,
    out=np.zeros_like(cm, dtype=float),
    where=row_totals != 0,
)
fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(cm_normalized, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(LABELS)), labels=LABELS, rotation=75, ha="right")
ax.set_yticks(range(len(LABELS)), labels=LABELS)
ax.set_xlabel("Nhãn dự đoán")
ax.set_ylabel("Nhãn thực tế")
ax.set_title("MLP Baseline E10 - Ma trận nhầm lẫn trên Validation")
fig.colorbar(image, ax=ax, label="Proportion per true class")
fig.tight_layout()
fig.savefig(IMAGE_DIR / "E10_validation_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

loss_curve = model.named_steps["classifier"].loss_curve_
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(loss_curve) + 1), loss_curve)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss huấn luyện")
ax.set_title("MLP Baseline E10 - Loss huấn luyện")
fig.tight_layout()
fig.savefig(IMAGE_DIR / "E10_training_loss.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Lưu kết quả Cross-Validation và Validation


In [ ]:
cv_results.to_csv(TABLE_DIR / "E10_cv_folds.csv", index=False, encoding="utf-8-sig")
cv_summary.to_csv(TABLE_DIR / "E10_cv_summary.csv", index=False, encoding="utf-8-sig")
cv_per_class.to_csv(TABLE_DIR / "E10_cv_per_class.csv", index=False, encoding="utf-8-sig")
cv_fig, cv_ax = plt.subplots(figsize=(7, 4))
cv_ax.bar(cv_results["fold"].astype(str), cv_results["macro_f1"])
cv_ax.set_xlabel("Lượt fold")
cv_ax.set_ylabel("Macro-F1")
cv_ax.set_ylim(0, 1)
cv_ax.set_title("MLP Baseline E10 - Macro-F1 qua 3 fold trên Train")
cv_fig.tight_layout()
cv_fig.savefig(IMAGE_DIR / "E10_cv_macro_f1.png", dpi=160, bbox_inches="tight")
plt.show()

metrics_df.to_csv(TABLE_DIR / "E10_metrics.csv", index=False, encoding="utf-8-sig")
per_class.to_csv(TABLE_DIR / "E10_validation_per_class.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(
    TABLE_DIR / "E10_validation_confusion_matrix.csv", encoding="utf-8-sig"
)

config = {
    "experiment": "E10_MLP_Baseline",
    "train_path": str(TRAIN_PATH),
    "validation_path": str(VALID_PATH),
    "cross_validation": {"method": "StratifiedKFold", "folds": N_SPLITS, "data": "chỉ tập Train có sẵn"},
    "test_used": False,
    "target": TARGET,
    "numeric_method": "StandardScaler",
    "categorical_method": "OneHotEncoder",
    "categorical_features": categorical_features,
    "train_samples": int(len(X_train)),
    "validation_samples": int(len(X_valid)),
    "classes": LABELS,
    "hidden_layer_sizes": [64],
    "max_iter": 60,
    "epochs_trained": int(model.named_steps["classifier"].n_iter_),
    "cv_macro_f1_mean": float(cv_summary.loc[cv_summary["metric"] == "macro_f1", "mean"].iloc[0]),
    "training_seconds": training_seconds,
    "random_state": RANDOM_STATE,
}
with (RUN_DIR / "E10_config.json").open("w", encoding="utf-8") as file:
    json.dump(config, file, ensure_ascii=False, indent=2)

with pd.ExcelWriter(RUN_DIR / "E10_MLP_Baseline.xlsx", engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="Train và Validation", index=False)
    cv_results.to_excel(writer, sheet_name="Các fold CV", index=False)
    cv_summary.to_excel(writer, sheet_name="Tổng hợp CV", index=False)
    cv_per_class.to_excel(writer, sheet_name="CV theo lớp", index=False)
    per_class.to_excel(writer, sheet_name="Validation theo lớp", index=False)
    pd.DataFrame([config]).to_excel(writer, sheet_name="Cấu hình", index=False)

print("Đã lưu kết quả vào:", RUN_DIR)